# YOLOE-26 Inference Setup 
##### Prepared by - Sourabh Sugandhi
##### Linkedin - https://www.linkedin.com/in/sourabh-sugandhi-386ab2159/
##### Github - https://github.com/sourabhsugandhi

### Environment Setup
This installs the necessary library to the local Colab VM and mounts your Drive.

In [ ]:
# Install Ultralytics (YOLO) to the local VM
!pip install -q ultralytics

import os
import cv2
import glob
from google.colab import drive
from ultralytics import YOLO

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

### Configuration & Path Setup
Define your input/output locations here so you only have to change them in one place.

In [ ]:
# Local paths for processing
INPUT_PATH = "/content/drive/MyDrive/YOLOE26/frames_4fps/*.jpg"
TEMP_TEXT_DIR = "/content/drive/MyDrive/YOLOE26/annotated_frames"
FINAL_VIDEO_NAME = "/content/drive/MyDrive/YOLOE26/aeroplane_segmentation.mp4"

# Create the output directory on your Drive
os.makedirs(TEMP_TEXT_DIR, exist_ok=True)

### Initialize Model & Run Prediction
This block runs the segmentation. Note that classes=[4] specifically targets airplanes in the COCO dataset.

In [ ]:
# --- Cell 3: Initialize YOLOE & Predict (Fixed) ---

# 1. Initialize the YOLOE-26 model specifically the "yoloe-26l-seg" architecture, which is part of the YOLO-MS or YOLOe family
model = YOLO("yoloe-26l-seg.pt")

# 2. PROMPT the model (Required for YOLOE open-vocabulary)
# This aligns the model's embeddings to the word 'aeroplane'
target_classes = ["aeroplane"]
model.set_classes(target_classes)

# 3. Run prediction
# We don't need 'classes=[4]' now because the prompt limits the model to only 'aeroplane'
results = model.predict(
    source=INPUT_PATH,
    save=True,      # This saves standard YOLO segmented outputs
    conf=0.15       # Lowering slightly to catch any misses
)

# 4. Get the directory where YOLO saved its internal 'predict' output
save_dir = results[0].save_dir
print(f"Detections processed! YOLO saved them at: {save_dir}")

### Add Text Overlay to Frames
We read the segmented images from the YOLO output folder, add the red "Prompt" text, and save them to our temporary folder.

In [ ]:
# Get all segmented images from the YOLO run directory
segmented_files = sorted(glob.glob(os.path.join(save_dir, "*.jpg")))
print(segmented_files)

print(f"Processing {len(segmented_files)} frames...")

for idx, file_path in enumerate(segmented_files):
    img = cv2.imread(file_path)
    if img is None: continue

    # Define text properties
    text = "Prompt : aeroplane"
    font = cv2.FONT_HERSHEY_SIMPLEX
    color = (0, 0, 255) # Red in BGR (OpenCV format)

    # Position text at the bottom
    height, width, _ = img.shape
    position = (50, height - 50)

    # Draw text on image
    cv2.putText(img, text, position, font, 1, color, 2, cv2.LINE_AA)

    # Save to our temp annotated folder
    out_name = f"frame_{idx:04d}.jpg"
    cv2.imwrite(os.path.join(TEMP_TEXT_DIR, out_name), img)

print("Text annotation complete.")

### Compile into MP4 Video
This takes the annotated frames and stitches them together at 4 FPS.

In [ ]:
# Get the annotated frames
frames = sorted(glob.glob(os.path.join(TEMP_TEXT_DIR, "*.jpg")))

if not frames:
    print("No frames found to compile.")
else:
    # Read first frame to get dimensions
    sample_img = cv2.imread(frames[0])
    h, w, _ = sample_img.shape

    # Initialize VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    video = cv2.VideoWriter(FINAL_VIDEO_NAME, fourcc, 4, (w, h))

    for f in frames:
        video.write(cv2.imread(f))

    video.release()
    print(f"Done! Video generated at: {FINAL_VIDEO_NAME}")

### Preview/Download (Optional)
This allows you to download the result directly to your local computer.

In [ ]:
from google.colab import files
files.download(FINAL_VIDEO_NAME)